# Reading floor plans, end to end

One notebook, one GPU, one run. It sets everything up, runs both models over all 25 test
plans, combines them, tunes the combination, and finishes with a table, a chart and a
picture of every flat.

Nothing to download halfway through. Nothing to upload. Run every cell top to bottom.

**Attach a GPU** in the sidebar kernel settings — a T4 is plenty, an A10G is quicker.
**About an hour**, most of it one compile.

---

### What it is doing, in one paragraph

Two models are involved and they are good at opposite halves of the job. A **room-finder**
looks at the whole plan and says *here is a kitchen, here is a bedroom* — it never reads the
printed text, so an unlabelled cupboard is no harder for it than a labelled bedroom, and it
keeps an open-plan kitchen-diner as one room. But it works on a small copy of the plan and
rounds every corner to a coarse grid, so its rooms are the right rooms in roughly the right
place. A **wall model** marks every pixel that is a wall, exactly, but has no idea which
enclosed space is a room. So: the room-finder's rooms become starting points, each one grows
outwards until it hits a wall on the wall model's map, and then the names printed on the plan
get attached to whichever room they were printed inside. Their rooms, our walls, our names.

### How everything is scored

Walk around the edge of every room the software found and ask, at each step: **is there a
wall here?** A room read correctly has its edge on the walls the whole way round. A room whose
edge cuts across open floor — because it stopped at the kitchen units, or ballooned out past
the building — scores low. We call it the **wall match**.

It has one blind spot worth knowing: a room that has grown out *past* the outer wall still has
its edge running along a wall, so it can score well and look plainly wrong. **The pictures at
the end are the arbiter, not the score.**

# Part 1 — Setup

Plumbing. Run these eight cells and don't read them.

### 1.1 · Is there a GPU?

In [ ]:
import subprocess

import torch

try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("no nvidia-smi on this runtime")
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU, then rerun"

### 1.2 · Get the two codebases

In [ ]:
import os
import subprocess
from pathlib import Path

# If you mounted a Modal Volume, put its path here — the results get copied there at
# the end and survive the machine being stopped. Leave it None and everything lives
# only as long as this kernel, and you download the zip from the sidebar instead.
VOLUME = None                      # e.g. Path("/mnt/my-volume")

# Everything this notebook makes lives here.
ROOT = Path("/root/plan-reading")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

R2S = ROOT / "Raster2Seq"
OURS = ROOT / "visit-it"


def clone(url, dest):
    if dest.exists():
        print(f"{dest.name}: already here")
        return
    r = subprocess.run(["git", "clone", "--depth", "1", url, str(dest)],
                       capture_output=True, text=True)
    print(f"{dest.name}: {'cloned' if not r.returncode else 'FAILED'}")
    if r.returncode:
        print(r.stderr[-600:])


clone("https://github.com/Cornell-VAILab/Raster2Seq.git", R2S)
clone("https://github.com/romainbigare/visit-it.git", OURS)
os.chdir(R2S)

### 1.3 · Install what is missing — carefully

Colab ships NumPy, OpenCV and Matplotlib as one set, all built against each other. Moving
any of them breaks the rest, so we pin NumPy where it already is and install only what is
genuinely absent.

In [ ]:
import shutil
import subprocess
import sys

import numpy

# Hold NumPy exactly where the image put it. OpenCV, SciPy and Matplotlib are all
# built against a particular NumPy, and moving it breaks every one of them.
PIN = f"numpy=={numpy.__version__}"
# Read off what the code actually imports, not guessed at. The vendored copy of
# detectron2 drags in a long tail of small packages -- cloudpickle, hydra, iopath,
# tabulate, termcolor, black -- that Colab happened to ship and a clean image does
# not. The preflight in 1.6 catches anything still missing.
NEEDED = ["opencv-python-headless", "scikit-image", "scipy", "shapely", "plotly",
          "imageio", "descartes", "omegaconf", "fvcore", "pycocotools",
          "segmentation-models-pytorch", "safetensors", "pytesseract", "timm",
          "cloudpickle", "hydra-core", "iopath", "tabulate", "termcolor", "black",
          "pyyaml", "yacs", "portalocker"]


def pip_install(*packages):
    """Install with NumPy held still. Use this for everything in this notebook —
    one unpinned install anywhere is enough to break the binary packages."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", PIN, *packages],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-1500:], r.stderr[-1500:])
        raise SystemExit(f"pip could not install {packages} alongside {PIN}")


print(f"holding {PIN}")
pip_install(*NEEDED)

# Tesseract is a program, not a Python package, and our own reading needs it to
# find the room names printed on the plan.
if shutil.which("tesseract") is None:
    subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "tesseract-ocr"],
                   capture_output=True)
HAVE_TESSERACT = shutil.which("tesseract") is not None
print("tesseract:", "installed" if HAVE_TESSERACT else
      "NOT AVAILABLE — step 0 will find fewer rooms than it should")

check = subprocess.run(
    [sys.executable, "-c",
     "import numpy, cv2, matplotlib, shapely, plotly, imageio, torch, skimage;"
     "print(numpy.__version__, cv2.__version__, matplotlib.__version__)"],
    capture_output=True, text=True)
if check.returncode:
    print(check.stderr[-1200:])
    raise SystemExit("something broke — restart the kernel and run from the top")
print("numpy / opencv / matplotlib:", check.stdout.strip(), "— all fine")

### 1.4 · Make the room-finder run, and build it

Raster2Seq was written for early-2024 libraries and four things have moved since. Each has
an exact modern equivalent, so these are renames, not rewrites. Then it compiles its two
GPU pieces — this is the slow cell, around five minutes.

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

import torch

REPO = R2S

CMAP_OLD = "from matplotlib.cm import get_cmap"
CMAP_MARK = "# patched: matplotlib >= 3.9"
CMAP_NEW = f"""try:
    {CMAP_OLD}
except ImportError:                      {CMAP_MARK} removed it
    from matplotlib import colormaps

    def get_cmap(name=None, lut=None):
        return colormaps[name]"""


def modernise(root: Path) -> dict:
    """Four renames. Each is guarded against matching its own output, so running
    this twice changes nothing the second time."""
    hits = {}
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.suffix in (".cu", ".cuh", ".cpp", ".h", ".hpp"):
            src = original = path.read_text()
            src, a = re.subn(r"(AT_DISPATCH_\w+\(\s*)(\w+)\.type\(\)",
                             r"\1\2.scalar_type()", src)          # Tensor::type() is gone
            src, b = re.subn(r"(\w+)\.type\(\)\.is_cuda\(\)", r"\1.is_cuda()", src)
            n = a + b
        elif path.suffix == ".py":
            src = original = path.read_text()
            n = 0
            if CMAP_MARK not in src and CMAP_OLD in src:          # matplotlib 3.9
                src = src.replace(CMAP_OLD, CMAP_NEW, 1)
                n += 1
            src, c = re.subn(                                     # pytorch 2.6
                r"torch\.load\(([^)]*?map_location=[^)]*?)\)",
                lambda m: (m.group(0) if "weights_only" in m.group(1)
                           else f"torch.load({m.group(1)}, weights_only=False)"),
                src)
            n += c
        else:
            continue
        if src != original:
            path.write_text(src)
            hits[str(path.relative_to(root))] = n
    return hits


edits = modernise(REPO)
print(f"patched {len(edits)} files" if edits else "nothing left to patch")

major, minor = torch.cuda.get_device_capability()
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
print(f"\nbuilding for {torch.cuda.get_device_name(0)} — about five minutes\n")


def build(where: Path) -> bool:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--no-build-isolation", "."],
                       cwd=where, capture_output=True, text=True, env=os.environ)
    print(f"{where.name}: {'built' if not r.returncode else 'FAILED'}")
    if r.returncode:
        for line in (r.stdout + r.stderr).splitlines()[-10:]:
            print("   ", line)
    return not r.returncode


build(REPO / "models" / "ops")
if not build(REPO / "diff_ras"):
    print("  (the second one is only used by a feature we switch off — this is fine)")

### 1.5 · Confirm the room-finder works

If the GPU piece did not build, there is a slower pure-Python version of the same
calculation. Identical answers, several times slower. Either is fine here.

In [ ]:
import importlib
import importlib.util
import site
import sys
from pathlib import Path

import torch

REPO = R2S
sys.path.insert(0, str(REPO))
FUNC = REPO / "models" / "ops" / "functions" / "ms_deform_attn_func.py"
UPSTREAM = "import MultiScaleDeformableAttention as MSDA"

SHIM = """
try:
    import MultiScaleDeformableAttention as MSDA
except ImportError:
    # The GPU piece did not build. The same calculation in plain PyTorch is
    # further down this file, so route through it. Inference only.
    class _PurePythonMSDA:
        @staticmethod
        def ms_deform_attn_forward(value, value_spatial_shapes, value_level_start_index,
                                   sampling_locations, attention_weights, im2col_step):
            return ms_deform_attn_core_pytorch(
                value, value_spatial_shapes, sampling_locations, attention_weights)

        @staticmethod
        def ms_deform_attn_backward(*_args, **_kwargs):
            raise RuntimeError("inference only without the compiled extension")

    MSDA = _PurePythonMSDA()
"""


def find_extension():
    """A package installed while this kernel was already running is invisible to it
    until the import caches are dropped. Worth trying before concluding it failed."""
    importlib.invalidate_caches()
    try:
        return importlib.import_module("MultiScaleDeformableAttention")
    except ImportError:
        pass
    site.main()
    roots = list(site.getsitepackages())
    for root in roots:
        for egg in Path(root).glob("MultiScaleDeformableAttention*.egg"):
            if str(egg) not in sys.path:
                sys.path.insert(0, str(egg))
    importlib.invalidate_caches()
    try:
        return importlib.import_module("MultiScaleDeformableAttention")
    except ImportError:
        return None


if find_extension() is not None:
    SLOW_PATH = False
    print("room-finder: fast GPU version")
else:
    src = FUNC.read_text()
    if "_PurePythonMSDA" not in src:
        FUNC.write_text(src.replace(UPSTREAM, SHIM.strip(), 1))
    for name in [m for m in sys.modules if "ms_deform_attn" in m]:
        del sys.modules[name]
    SLOW_PATH = True
    print("room-finder: slower plain-PyTorch version (same answers)")

spec = importlib.util.spec_from_file_location("_check", FUNC)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
shapes = torch.as_tensor([[8, 6], [4, 3]], dtype=torch.long, device="cuda")
sizes = shapes[:, 0] * shapes[:, 1]
with torch.no_grad():
    out = mod.MSDeformAttnFunction.apply(
        torch.randn(2, int(sizes.sum()), 8, 32, device="cuda"), shapes,
        torch.cat([sizes.new_zeros(1), sizes.cumsum(0)[:-1]]),
        torch.rand(2, 7, 8, 2, 4, 2, device="cuda"),
        torch.rand(2, 7, 8, 2, 4, device="cuda"), 64)
assert tuple(out.shape) == (2, 7, 256) and torch.isfinite(out).all()
print("   checked — it computes the right thing")

# The model's own import chain, exercised before we spend five minutes discovering
# it at the first run. A missing module names itself in the error, so install it and
# try again rather than making you paste the traceback back to me.
PIP_NAME = {"cv2": "opencv-python-headless", "skimage": "scikit-image",
            "PIL": "pillow", "sklearn": "scikit-learn", "yaml": "pyyaml",
            "hydra": "hydra-core", "shapely": "shapely", "pycocotools": "pycocotools",
            "google": "protobuf", "cairosvg": "cairosvg", "drawsvg": "drawsvg",
            "svgpathtools": "svgpathtools", "svgwrite": "svgwrite",
            "plyfile": "plyfile", "webcolors": "webcolors"}


def preflight(rounds=12):
    tried = set()
    for _ in range(rounds):
        r = subprocess.run([sys.executable, "-c", "import predict"],
                           cwd=str(R2S), capture_output=True, text=True)
        if r.returncode == 0:
            return True
        err = r.stderr.strip().splitlines()
        missing = next((ln.split("'")[1] for ln in reversed(err)
                        if "ModuleNotFoundError: No module named" in ln), None)
        if not missing:
            print("   the model cannot start, and it is not a missing package:")
            print("\n".join(err[-12:]))
            return False
        pkg = PIP_NAME.get(missing, missing)
        if missing in tried:
            # Installing it did not make it importable, so the name we guessed is
            # wrong. Say which, rather than reinstalling it until the loop runs out.
            print(f"   installed {pkg}, but '{missing}' still will not import.")
            print(f"   Add the right pip name for '{missing}' to PIP_NAME above "
                  f"and re-run this cell.")
            return False
        tried.add(missing)
        print(f"   missing {missing} → installing {pkg}")
        try:
            pip_install(pkg)
        except SystemExit:
            print(f"   could not install {pkg} — stopping here")
            return False
    print(f"   still missing things after {rounds} rounds — stopping")
    return False


print()
print("checking the model can start")
READY = preflight()
print("   ready" if READY else "   NOT ready — the steps below will report nothing")

### 1.6 · The 25 test plans

Downloaded straight from the addresses recorded in our own test set, so nothing needs
uploading and nothing needs to exist on your machine.

In [ ]:
import json
import ssl
import urllib.request
from pathlib import Path

PLANS = ROOT / "plans" / "original"
PLANS.mkdir(parents=True, exist_ok=True)
golden = json.loads((OURS / "data" / "golden" / "golden_set.json").read_text())

ctx = ssl.create_default_context()
missing = []
for listing in golden["listings"]:
    plans = listing.get("floorplans") or []
    if not plans:
        missing.append(listing["listing_id"])
        continue
    dest = PLANS / f"{listing['listing_id']}.png"
    if dest.exists():
        continue
    try:
        req = urllib.request.Request(plans[0]["url"], headers={"User-Agent": "Mozilla/5.0"})
        dest.write_bytes(urllib.request.urlopen(req, context=ctx, timeout=60).read())
    except Exception as exc:
        print("  could not fetch", listing["listing_id"], exc)
        missing.append(listing["listing_id"])

IDS = sorted(p.stem for p in PLANS.glob("*.png"))
print(f"{len(IDS)} plans ready" + (f"  ·  {len(missing)} listings have no plan" if missing else ""))

### 1.7 · The wall map, and the score

One model in this notebook does nothing but say, for every pixel, whether it is a wall.
That is what we score against — because "is this edge on *some* line?" cannot tell a wall
from a kitchen cabinet, and cabinets are exactly what has been going wrong.

This cell downloads that model and defines the score. Read the two short functions if you
like; you do not need to.

In [ ]:
import sys
import urllib.request
import warnings
from pathlib import Path

import numpy as np
from PIL import Image

sys.path.insert(0, str(OURS))
warnings.filterwarnings("ignore")

from pipeline.floorplan import ocr as ocr_mod
from pipeline.floorplan import preprocess, vectorise, wallnet

(OURS / "models").mkdir(parents=True, exist_ok=True)
wallnet.MODEL_PATH = OURS / "models" / "plan_walls.safetensors"
if not wallnet.MODEL_PATH.exists():
    urllib.request.urlretrieve(wallnet.MODEL_URL, wallnet.MODEL_PATH)
assert wallnet.available(), "the wall model did not load"

# Two pictures of every plan, and a wall map for each.
#
#   "original"  — the file as the agent published it. The room-finder reads these.
#   "geometry"  — our own pipeline straightens and shrinks the plan before working on
#                 it, so its answers are in *those* pixels.
#
# An answer has to be scored against the wall map of the picture it was drawn on. Score
# it against the other one and every edge lands a couple of percent out, which looks
# exactly like the software being wrong when it is not.
WALLS, GEOM_IMG, GEOM_SIZE = {}, {}, {}
print("building a wall map for each plan (about a minute)")
for lid in IDS:
    rgb = np.array(Image.open(PLANS / f"{lid}.png").convert("RGB"))
    ink, _ = preprocess.ink_mask(rgb)
    WALLS[(lid, "original")] = wallnet.barrier(rgb, ink, [])

    pi = preprocess.prepare(PLANS / f"{lid}.png")
    text = ocr_mod.read(pi.rgb)
    WALLS[(lid, "geometry")] = wallnet.barrier(pi.rgb, pi.ink, text.words, pi.wall_half_px)
    GEOM_IMG[lid] = pi.rgb
    GEOM_SIZE[lid] = (pi.ink.shape[1], pi.ink.shape[0])
print("done")


def wall_match(reading, space="original"):
    """Share of every room's edge that lies on a wall. One number per plan."""
    per_plan = {}
    for lid, rec in reading.items():
        ref = WALLS.get((lid, space))
        if ref is None or not ref.any():
            continue
        polys = [r["polygon_px"] for r in rec["rooms"] if len(r["polygon_px"]) >= 3]
        scores = wallnet.outline_on_wall(polys, ref)
        if scores:
            per_plan[lid] = float(np.median(scores))
    return per_plan


LADDER = []


def record(name, reading, space, note):
    """Score a reading, add it to the ladder, and say how it did."""
    per_plan = wall_match(reading, space)
    if not per_plan:
        print(f"{name}: produced nothing to score")
        return None
    score = float(np.median(list(per_plan.values())))
    rooms = sum(len(r["rooms"]) for r in reading.values())
    entry = {"name": name, "score": score, "plans": len(per_plan), "rooms": rooms,
             "note": note, "space": space, "per_plan": per_plan}
    LADDER.append(entry)
    prev = LADDER[-2]["score"] if len(LADDER) > 1 else None
    best = max(e["score"] for e in LADDER[:-1]) if len(LADDER) > 1 else None
    print(f"  {name}")
    print(f"  wall match  {score:.0%}   ·  {rooms} rooms found across "
          f"{len(per_plan)} plans")
    if prev is not None:
        arrow = "better" if score > prev + 0.005 else ("worse" if score < prev - 0.005 else "no change")
        print(f"  previous step was {prev:.0%}  →  {arrow}")
    if best is not None and score > best + 0.005:
        print(f"  best so far")
    return entry

### 1.8 · Running the room-finder

One function, used by every step below. It writes the plans somewhere, runs the model over
them, and hands back the rooms it found in the plan's own pixels.

In [ ]:
import json
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import numpy as np
from PIL import Image

# Every checkpoint publishes the settings it was trained with, so read them rather
# than hand-writing flags. Hand-writing them is how the "sharper model" step failed
# first time round: it was run at 32 coordinate bins with 12 room types when that
# checkpoint wants 32 bins and 13, and against the 256-pixel weights while asking
# for a 512-pixel model.
HUB = "https://huggingface.co/haopt/Raster2Seq/resolve/main"

# What the room types are called, per training set.
CC5K_NAMES = {0: "Outdoor", 1: "Kitchen", 2: "Living Room", 3: "Bed Room", 4: "Bath",
              5: "Entry", 6: "Storage", 7: "Garage", 8: "Undefined"}
CC5K_APERTURES = {9, 10}                       # window and door, kept out of the rooms
R2G_NAMES = {0: "unknown", 1: "living room", 2: "kitchen", 3: "bedroom", 4: "bathroom",
             5: "restroom", 6: "balcony", 7: "closet", 8: "corridor",
             9: "washing room", 10: "PS", 11: "outside"}
R2G_APERTURES = set()                          # this one marks no doors or windows

_CONFIGS = {}


def hub_subfolder(key):
    """Where that checkpoint lives on the hub.

    The name you pass and the folder it sits in are not always the same -- the
    512-pixel one is called `raster2graph-512` and stored under `Raster2Graph-512`
    -- so read the repository's own table rather than keeping a second copy of it
    here that can drift.
    """
    try:
        sys.path.insert(0, str(R2S))
        import raster2seq_hub
        canonical = raster2seq_hub.normalize_checkpoint_name(key)
        return canonical, str(raster2seq_hub.CHECKPOINTS[canonical]["subfolder"])
    except Exception:
        return key, key


def checkpoint_config(key):
    """The settings that checkpoint was trained with, straight from the hub."""
    if key not in _CONFIGS:
        canonical, folder = hub_subfolder(key)
        with urllib.request.urlopen(f"{HUB}/{folder}/config.json", timeout=60) as r:
            _CONFIGS[key] = json.load(r)
        _CONFIGS[key]["_alias"] = canonical
    return _CONFIGS[key]


def flags_from(config):
    """Turn a checkpoint's own inference_args into command-line flags."""
    args = dict(config["inference_args"])
    for drop in ("dataset_root", "eval_set", "output_dir"):
        args.pop(drop, None)
    out = []
    for k, v in args.items():
        if v is True:
            out.append(f"--{k}")
        elif v is False or v is None:
            continue
        else:
            out += [f"--{k}", str(v)]
    return out, int(args.get("image_size", 256)), args.get("dataset_name", "cubicasa")


def undo_letterbox(poly, src_w, src_h, size):
    """The model works on a small square copy with grey bars; put the coordinates back."""
    scale = min(size / src_h, size / src_w)
    new_h, new_w = int(src_h * scale), int(src_w * scale)
    left, top = (size - new_w) // 2, (size - new_h) // 2
    p = np.asarray(poly, dtype=float).reshape(-1, 2)
    return np.stack([(p[:, 0] - left) / scale, (p[:, 1] - top) / scale], axis=1)


def find_rooms(images_dir, tag, checkpoint="cubicasa5k"):
    """Run the room-finder over a directory of plans, using that checkpoint's own
    settings. Returns {listing: {"rooms": [...]}} in each plan's own pixels."""
    if not READY:
        print("  skipped: the room-finder could not start — see the preflight in 1.5")
        return {}
    try:
        config = checkpoint_config(checkpoint)
    except Exception as exc:
        print(f"  could not read the settings for '{checkpoint}': {exc}")
        return {}
    flags, size, dataset = flags_from(config)
    names = R2G_NAMES if dataset == "r2g" else CC5K_NAMES
    apertures = R2G_APERTURES if dataset == "r2g" else CC5K_APERTURES
    print(f"  {config['name']}  ·  {size}px  ·  "
          f"scores {config['metrics']['room_f1']} on its own test set")

    out_dir = ROOT / "runs" / tag
    if out_dir.exists():
        shutil.rmtree(out_dir)
    t0 = time.time()
    cmd = ["python", "predict.py", f"--dataset_root={images_dir}",
           f"--output_dir={out_dir}", f"--checkpoint=hf:{config['_alias']}", *flags]
    r = subprocess.run(cmd, cwd=str(R2S), capture_output=True, text=True)
    if r.returncode:
        print(f"  the model failed on '{tag}':")
        for line in (r.stdout + r.stderr).splitlines()[-8:]:
            print("     ", line)
        return {}

    reading = {}
    for jf in sorted(out_dir.rglob("jsons/*.json")):
        lid = jf.stem
        # Undo the letterbox against the picture the model was actually shown, not
        # the original plan. They are the same size for a cleaned-up or a tilted
        # copy -- but a quarter-tile is not, and sizing that one from the whole
        # plan puts every room it finds a third of the way out.
        shown = Path(images_dir) / f"{lid}.png"
        src = Image.open(shown if shown.exists() else PLANS / f"{lid}.png")
        rooms = []
        for inst in json.loads(jf.read_text()):
            cid = inst["category_id"]
            if cid in apertures or cid not in names:
                continue
            rooms.append({"polygon_px": undo_letterbox(inst["segmentation"],
                                                       src.width, src.height, size).tolist(),
                          "label": names[cid]})
        reading[lid] = {"rooms": rooms}
    print(f"  ran in {time.time() - t0:.0f}s")
    return reading

---

# Part 2 — Reading the plans

Six readings, each measured on the same 25 plans and compared to the one before.

## Step 0 · What we had before either model

Our own reading: the wall model, with rooms grown from the room names printed on the plan.
Its weakness is the starting points — a room with no printed name has none.

In [ ]:
ours = {}
for lid in IDS:
    try:
        pi = preprocess.prepare(PLANS / f"{lid}.png")
        text = ocr_mod.read(pi.rgb)
        v = vectorise.segment(pi, text)
    except Exception as exc:
        print(f"  {lid}: {type(exc).__name__}: {exc}")
        continue
    ours[lid] = {"rooms": [{"polygon_px": r.polygon_px, "label": r.label or ""}
                           for r in v.rooms]}

record("What we have today", ours, "geometry",
       "our software: wall model, rooms grown from the printed room names")

## Step 1 · The room-finder, on the plans as published

In [ ]:
raw = find_rooms(PLANS, "raw", checkpoint="cubicasa5k")
record("Second opinion, plans as published", raw, "original",
       "Raster2Seq on the original files")

## Step 2 · The room-finder, on a cleaned-up picture

It learned on plans drawn as black lines on white paper with no text. Ours are tinted,
colour-filled and covered in writing, so it gets a tidied copy: page levelled to white,
every word painted out.

In [ ]:
import cv2
import pytesseract

CLEAN = ROOT / "plans" / "cleaned"
CLEAN.mkdir(parents=True, exist_ok=True)


def level_to_white(rgb):
    """Whatever colour the page is, make it white; whatever the darkest ink is, black."""
    lum = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    page = int(np.argmax(np.bincount(lum.ravel(), minlength=256)))
    if page < 110:                       # white lines on a dark page: flip first
        lum, page = 255 - lum, 255 - page
    floor = float(np.percentile(lum, 1))
    if page - floor < 20:
        return cv2.cvtColor(lum, cv2.COLOR_GRAY2RGB)
    out = np.clip((lum.astype(np.float32) - floor) * (255.0 / (page - floor)), 0, 255)
    return cv2.cvtColor(out.astype(np.uint8), cv2.COLOR_GRAY2RGB)


def paint_out_words(rgb):
    data = pytesseract.image_to_data(rgb, output_type=pytesseract.Output.DICT)
    out, h, w = rgb.copy(), *rgb.shape[:2]
    for i, conf in enumerate(data["conf"]):
        if float(conf) < 40 or not data["text"][i].strip():
            continue
        x, y = data["left"][i] - 2, data["top"][i] - 2
        out[max(0, y):min(h, y + data["height"][i] + 4),
            max(0, x):min(w, x + data["width"][i] + 4)] = 255
    return out


for lid in IDS:
    dest = CLEAN / f"{lid}.png"
    if not dest.exists():
        rgb = np.array(Image.open(PLANS / f"{lid}.png").convert("RGB"))
        Image.fromarray(paint_out_words(level_to_white(rgb))).save(dest)

cleaned = find_rooms(CLEAN, "cleaned")
record("Second opinion, cleaned-up picture", cleaned, "original",
       "page levelled to white, all text painted out")

In [ ]:
# See what it is now looking at
import matplotlib.pyplot as plt

for lid in IDS[:3]:
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    for a, d, t in zip(ax, (PLANS, CLEAN), ("as published", "cleaned up for the model")):
        a.imshow(Image.open(d / f"{lid}.png")); a.axis("off"); a.set_title(t, fontsize=10)
    plt.tight_layout(); plt.show()

## Step 3 · Try the sharper model

The authors publish several trained models. The one we have used so far works at
**256 pixels**; another works at **512**, so the corners it reports can land twice as
precisely. It was trained on a different collection of plans, with a different list of
room types, which may suit ours better or worse.

Each model comes with a record of the settings it was trained with, and this notebook
reads that record rather than assuming — the number of coordinate bins, the number of room
types and the working resolution all differ between them, and getting any one of them
wrong means the model refuses to load at all.

In [ ]:
# The authors also publish this one trained at 512 pixels instead of 256, on a
# different set of plans. Its own settings come with it, so nothing here is guessed.
sharper = find_rooms(CLEAN, "sharper", checkpoint="raster2graph-512")
if sharper:
    record("The sharper 512-pixel model", sharper, "original",
           "trained at twice the resolution, on a different set of plans")
else:
    print("  this step did not produce anything — carrying on with the best so far")

## Step 4 · Show it the plan in pieces

The model shrinks the whole plan to a small square before looking at it, so a cupboard or a
WC can end up only a few pixels across and disappear.

So we cut each plan into four overlapping quarters, run the model on each at full size, and
put the answers back together — keeping whichever version of an overlapping room covers
more floor.

This is the least likely of the six to work: a quarter of a flat is not a flat, and the
model was trained on whole ones.

In [ ]:
from itertools import product

TILES = ROOT / "plans" / "tiles"


def cut_into_tiles(overlap=0.25):
    if TILES.exists():
        shutil.rmtree(TILES)
    TILES.mkdir(parents=True)
    boxes = {}
    for lid in IDS:
        im = Image.open(CLEAN / f"{lid}.png").convert("RGB")
        w, h = im.size
        tw, th = int(w * (0.5 + overlap / 2)), int(h * (0.5 + overlap / 2))
        for i, (cx, cy) in enumerate(product((0, w - tw), (0, h - th))):
            name = f"{lid}__{i}"
            im.crop((cx, cy, cx + tw, cy + th)).save(TILES / f"{name}.png")
            boxes[name] = (cx, cy, tw, th)
    return boxes


def merge_tiles(tile_reading, boxes):
    """Put every tile's rooms back on the full plan, then drop the duplicates."""
    from shapely.geometry import Polygon
    merged = {}
    for name, rec in tile_reading.items():
        lid, _, _idx = name.partition("__")
        cx, cy, _tw, _th = boxes[name]
        for room in rec["rooms"]:
            p = np.asarray(room["polygon_px"], float) + [cx, cy]
            merged.setdefault(lid, []).append({"polygon_px": p.tolist(),
                                               "label": room["label"]})
    out = {}
    for lid, rooms in merged.items():
        polys = []
        for room in sorted(rooms, key=lambda r: -_area(r["polygon_px"])):
            try:
                g = Polygon(room["polygon_px"]).buffer(0)
            except Exception:
                continue
            if g.is_empty or g.area <= 0:
                continue
            # a room already covered by a bigger one from another tile is the same room
            if any(g.intersection(k).area > 0.5 * g.area for k in polys):
                continue
            polys.append(g)
            out.setdefault(lid, {"rooms": []})["rooms"].append(room)
    return out


def _area(poly):
    p = np.asarray(poly, float)
    return abs(float(np.dot(p[:, 0], np.roll(p[:, 1], 1)) -
                     np.dot(p[:, 1], np.roll(p[:, 0], 1))) / 2)


tiled = {}
try:
    boxes = cut_into_tiles()
    # find_rooms names results after the file, and tiles are named "<listing>__<n>"
    tile_reading = find_rooms(TILES, "tiles")
    tiled = merge_tiles(tile_reading, boxes)
except Exception as exc:
    print("  did not run:", type(exc).__name__, exc)

if tiled:
    record("Shown the plan in four pieces", tiled, "original",
           "four overlapping quarters, answers merged")
else:
    print("  this step did not produce anything — carrying on with the best so far")

## Step 5 · Ask three ways, keep what agrees

The plan as published, cleaned up, and cleaned up but slightly rotated. A room only one
version finds is usually a mistake; one all three find is usually real.

In [ ]:
from shapely.geometry import Polygon

ROT = ROOT / "plans" / "rotated"
ROT.mkdir(parents=True, exist_ok=True)
ANGLE = 2                                  # degrees, anticlockwise

for lid in IDS:
    dest = ROT / f"{lid}.png"
    if not dest.exists():
        # Same size as the plan, so the answers come back in the same frame, only
        # turned. Letting the picture grow to fit the corners would change the
        # frame and put every room a couple of percent out.
        Image.open(CLEAN / f"{lid}.png").rotate(ANGLE, fillcolor=(255, 255, 255)).save(dest)


def unrotate(poly, w, h):
    """Turn an answer found on the tilted copy back onto the plan itself."""
    t = np.deg2rad(ANGLE)
    back = np.array([[np.cos(t), np.sin(t)], [-np.sin(t), np.cos(t)]])
    middle = np.array([w / 2.0, h / 2.0])
    return ((np.asarray(poly, float) - middle) @ back.T + middle).tolist()


votes = {}
try:
    rotated = find_rooms(ROT, "rotated")
    for lid, rec in rotated.items():
        w, h = Image.open(PLANS / f"{lid}.png").size
        for room in rec["rooms"]:
            room["polygon_px"] = unrotate(room["polygon_px"], w, h)
    for source in (raw, cleaned, rotated):
        for lid, rec in source.items():
            votes.setdefault(lid, []).extend(rec["rooms"])
except Exception as exc:
    print("  did not run:", exc)

consensus = {}
for lid, rooms in votes.items():
    shapes, kept = [], []
    for room in sorted(rooms, key=lambda r: -_area(r["polygon_px"])):
        try:
            g = Polygon(room["polygon_px"]).buffer(0)
        except Exception:
            continue
        if g.is_empty or g.area <= 0:
            continue
        hits = [i for i, k in enumerate(shapes)
                if g.intersection(k).area > 0.5 * min(g.area, k.area)]
        if hits:
            continue                       # same room, already have the larger version
        shapes.append(g)
        kept.append(room)
    if kept:
        consensus[lid] = {"rooms": kept}

if consensus:
    record("Asked three ways, kept the agreement", consensus, "original",
           "as published + cleaned + rotated, overlapping rooms merged")

---

# Part 3 — Putting the two together

The room-finder's rooms become starting points; each grows out until it hits a wall on the
wall model's map; then the names printed on the plan are attached to whichever room they were
printed inside.

This is the same code the pipeline runs — `vectorise.segment_from_room_seeds` — so whatever
wins here is what ships, with no reimplementation in between.

In [ ]:
import time

# Everything a plan needs that does not depend on which run we grow from:
# straightening it, reading its text, and the wall model's map. About ten seconds a
# plan, done once — after this, trying another way of combining takes seconds.
BASE = {}
t0 = time.time()
for lid in IDS:
    walls = WALLS.get((lid, "geometry"))
    if walls is None or not walls.any():
        continue
    pi = preprocess.prepare(PLANS / f"{lid}.png")
    BASE[lid] = {"pi": pi, "text": ocr_mod.read(pi.rgb), "walls": walls}
print(f"prepared {len(BASE)} plans in {time.time() - t0:.0f}s")

# Every way the room-finder was run above, under the name it was scored by.
NAME_TO_RUN = {"Second opinion, plans as published": raw,
               "Second opinion, cleaned-up picture": cleaned,
               "The sharper 512-pixel model": sharper,
               "Shown the plan in four pieces": tiled,
               "Asked three ways, kept the agreement": consensus}
finder_scored = [e for e in LADDER if e["space"] == "original"]
best_finder = max(finder_scored, key=lambda e: e["score"]) if finder_scored else None
FINDER_ALONE = NAME_TO_RUN.get(best_finder["name"]) if best_finder else cleaned
PREDICTIONS = FINDER_ALONE          # reassigned below to whichever run grows best


def seeds_from(reading):
    """One room-finder run, turned into starting points on the straightened plan."""
    out = {}
    for lid, base in BASE.items():
        rooms = [r for r in ((reading or {}).get(lid) or {}).get("rooms", [])
                 if len(r.get("polygon_px") or []) >= 3]
        if not rooms:
            continue
        out[lid] = {
            "seeds": vectorise.roomfinder.to_geometry(
                [r["polygon_px"] for r in rooms], base["pi"]),
            "names": [str(r.get("label") or "").lower() for r in rooms],
        }
    return out


COMBINED, SEED_SOURCE = {}, {}


def combine(name, note, reading=None, **kwargs):
    """Grow one run's rooms out to the walls, name them from the plan, score it."""
    source = PREDICTIONS if reading is None else reading
    prepared = seeds_from(source)
    result = {}
    for lid, seeded in prepared.items():
        base = BASE[lid]
        try:
            v = vectorise.segment_from_room_seeds(
                base["pi"], base["text"], seeded["seeds"], mask=base["walls"],
                fallback_labels=seeded["names"], **kwargs)
        except Exception as exc:                                   # noqa: BLE001
            print(f"  {lid}: {type(exc).__name__}: {exc}")
            continue
        if v is None or not v.rooms:
            continue
        result[lid] = {"rooms": [{"polygon_px": r.polygon_px, "label": r.label or "",
                                  "seeded_by": r.seeded_by} for r in v.rooms]}
    entry = record(name, result, "geometry", note)
    if entry is not None:
        printed = sum(1 for v in result.values() for r in v["rooms"]
                      if r["seeded_by"] == "caption")
        entry["printed"], entry["kwargs"] = printed, kwargs
        COMBINED[name], SEED_SOURCE[name] = result, source
        print(f"  {printed} of {entry['rooms']} rooms named from the plan itself")
    return entry


## Which run is worth growing

The run that scores best on its own is not necessarily the best one to grow from — a run can
score well by finding fewer, tidier rooms, and it is the *rooms* we are after. So every one
of them gets grown out to the walls and scored, and the winner is what the two settings
below are then tuned on.

In [ ]:
# The run that scores best on its own is not necessarily the best to grow from: a
# run can score well by finding fewer, tidier rooms, and it is the *rooms* we want.
# So grow every one of them and let the score say.
for run_name, reading in NAME_TO_RUN.items():
    if not reading:
        continue
    combine(f"Both models · {run_name.lower()}", f"grown from: {run_name}", reading=reading)

joined = [e for e in LADDER if e["name"].startswith("Both models · ")]
if joined:
    best_join = max(joined, key=lambda e: e["score"])
    PREDICTIONS = SEED_SOURCE[best_join["name"]]
    print(f"\ngrowing from now on: {best_join['name'][14:]}  "
          f"({best_join['score']:.0%}, {best_join['rooms']} rooms)")
else:
    print("\nnothing to grow — the room-finder produced no rooms to start from")


## The two settings that govern the combination

Growing a predicted room out to the walls has two settings, and both are swept here rather
than assumed.

**How far in to pull the starting point.** The predicted outline is coarse, so its edge lands
on a wall or slightly past it. A starting point touching a wall lets the room grow straight
through into its neighbour, so each one is shrunk first. Too little and rooms leak; too much
and small rooms vanish.

**Where a room may go.** Today a room may grow anywhere inside the *convex hull* of the
drawing — and the hull of an L-shaped or bay-fronted plan includes the garden in the crook of
the L, on average **seven times** the real footprint. That is why rooms sometimes shoot out
into the margin as spikes. The alternative follows the walls' real outline, and falls back to
today's behaviour on plans where it cannot find one.

In [ ]:
combine("… starting points pulled in less (25%)",
        "closer to the model's own outline — better for small rooms, riskier for leaks",
        seed_core=0.25)
combine("… starting points pulled in more (65%)",
        "a small core in the middle of each room — safe against leaks, hard on small rooms",
        seed_core=0.65)
combine("… rooms kept inside the building",
        "the walls' real outline instead of the drawing's convex hull",
        confine=True)

sizes = [e for e in LADDER if "starting points pulled" in e["name"]]
best_size = max(sizes, key=lambda e: e["score"]) if sizes else None
core = best_size["kwargs"].get("seed_core") if best_size else None
combine(f"… both{'' if not core else f' ({core:.0%} + inside)'}",
        "the best starting-point size, and confined to the building",
        **({"seed_core": core} if core else {}), confine=True)


---

# Part 4 — The report

### The table

In [ ]:
best = max(LADDER, key=lambda e: e["score"])
start = LADDER[0]

print(f'{"":<3}{"reading":<54}{"wall match":>11}{"vs start":>10}{"rooms":>7}')
print("-" * 86)
for i, e in enumerate(LADDER):
    delta = "—" if i == 0 else f'{e["score"] - start["score"]:+.0%}'
    mark = " ←" if e is best else ""
    print(f'{i:<3}{e["name"][:53]:<54}{e["score"]:>10.0%}{delta:>10}{e["rooms"]:>7}{mark}')
print("-" * 86)
print(f'\nbest: {best["name"]} — {best["score"]:.0%}, from {start["score"]:.0%} at the start')
print(f'({best["note"]})')

# A higher score with fewer rooms is not obviously a win: this score asks whether a
# room's edge sits on a wall, and deleting an awkward room raises it just as surely
# as fixing one does. Only the pictures can tell those apart.
lost = start["rooms"] - best["rooms"]
if lost > 0.1 * start["rooms"]:
    print(f'\n  CAREFUL: it also found {lost} fewer rooms ({best["rooms"]} against '
          f'{start["rooms"]}).')
    print('  A reading that deletes an awkward room scores better for doing so.')
    print('  Look at the pictures below before taking this one.')

if best.get("kwargs"):
    print('\nTo make it the default, set these in pipeline/floorplan/vectorise.py:')
    for k, v in best["kwargs"].items():
        const = "SEED_CORE_FRACTION" if k == "seed_core" else "CONFINE_TO_FOOTPRINT"
        print(f'    {const} = {v!r}')

### The chart

In [ ]:
import matplotlib.pyplot as plt

# One measure, one series, so one colour for every bar — the length is the message,
# and a second colour would only repeat it.
BAR = "#2a78d6"
INK, MUTED, RULE = "#0b0b0b", "#52514e", "#d8d8d2"

names = [e["name"] for e in LADDER]
scores = [e["score"] for e in LADDER]
y = np.arange(len(LADDER))

fig, ax = plt.subplots(figsize=(11, 0.62 * len(LADDER) + 2.1))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")
ax.barh(y, scores, height=0.6, color=BAR, zorder=3)
ax.axvline(scores[0], color=MUTED, linestyle="--", linewidth=1, zorder=2)
ax.text(scores[0], -0.78, "where we started", color=MUTED, fontsize=9,
        ha="center", va="bottom")

for i, s in enumerate(scores):
    ax.text(s + 0.012, i, f"{s:.0%}" + ("   best" if LADDER[i] is best else ""),
            va="center", fontsize=10, color=INK,
            fontweight="bold" if LADDER[i] is best else "normal")

ax.set_yticks(y); ax.set_yticklabels(names, fontsize=10, color=INK)
ax.invert_yaxis()
ax.set_xlim(0, max(scores) * 1.22)
ax.set_xticks(np.arange(0, 1.01, 0.2))
ax.set_xticklabels([f"{v:.0%}" for v in np.arange(0, 1.01, 0.2)], color=MUTED, fontsize=9)
ax.set_xlabel("share of every room's edge that lands on a wall", color=MUTED, fontsize=10)
ax.grid(axis="x", color=RULE, linewidth=1, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(RULE)
ax.tick_params(length=0)
ax.set_title(f"Every reading, on the same {LADDER[0]['plans']} plans",
             fontsize=13, color=INK, loc="left", pad=22)
plt.tight_layout(); plt.show()

### Every flat: ours, theirs, and the two together

| panel | what it is |
|---|---|
| 1 | the plan, untouched |
| 2 | **ours** — the wall model, rooms grown from the printed names |
| 3 | **theirs** — the best the room-finder managed alone |
| 4 | **both** — the best combination above |

Read the shapes, not the numbers:

- is an **open-plan kitchen-and-living-room one room**, or two?
- are the **bathroom, WC, hallway and cupboards** there at all?
- do any rooms **shoot out past the building** into the margin? The score cannot see that.

In [ ]:
import colorsys

from matplotlib.patches import Polygon as MplPoly

ORIGINAL_IMG = {lid: np.array(Image.open(PLANS / f"{lid}.png").convert("RGB")) for lid in IDS}
combined_best = max((e for e in LADDER if e["name"] in COMBINED),
                    key=lambda e: e["score"], default=None)


def hue(i):
    return colorsys.hsv_to_rgb((i * 0.61803) % 1.0, 0.55, 0.95)


def draw(ax, img, rooms, title):
    ax.imshow(img); ax.axis("off"); ax.set_title(title, fontsize=10)
    for i, room in enumerate(rooms):
        p = np.asarray(room["polygon_px"])
        if len(p) < 3:
            continue
        c = hue(i)
        ax.add_patch(MplPoly(p, closed=True, facecolor=c + (0.35,), edgecolor=c, linewidth=2))
        if room.get("label"):
            ax.text(*p.mean(axis=0), room["label"], ha="center", va="center", fontsize=7.5,
                    weight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75))


def panel(entry, reading, lid):
    rooms = (reading or {}).get(lid, {"rooms": []})["rooms"]
    img = GEOM_IMG[lid] if (entry and entry["space"] == "geometry") else ORIGINAL_IMG[lid]
    score = entry["per_plan"].get(lid) if entry else None
    caption = f"{len(rooms)} rooms" + (f" · {score:.0%}" if score is not None else "")
    return img, rooms, caption


for lid in IDS:
    fig, axes = plt.subplots(1, 4, figsize=(23, 6.2))
    axes[0].imshow(ORIGINAL_IMG[lid]); axes[0].axis("off")
    axes[0].set_title(f"{lid} — the plan", fontsize=10)
    for ax, (entry, reading, label) in zip(axes[1:], [
            (LADDER[0], ours, "ours"),
            (best_finder, FINDER_ALONE, "the room-finder alone"),
            (combined_best, COMBINED.get(combined_best["name"]) if combined_best else None,
             "both together")]):
        img, rooms, caption = panel(entry, reading, lid)
        draw(ax, img, rooms, f"{label} · {caption}")
    plt.tight_layout(); plt.show()

### Save the results

In [ ]:
import shutil

OUT = ROOT / "results"
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()
(OUT / "every_reading.json").write_text(json.dumps(
    [{k: v for k, v in e.items() if k != "kwargs"} | {"kwargs": e.get("kwargs", {})}
     for e in LADDER], indent=1))
SCORED = {e["name"]: e["score"] for e in LADDER}

# Say what each saved reading is, rather than leaving the importer to guess from its
# name. Only the room-finder's own rooms are worth importing: ours is what stage 5
# already produces, and a combined one is stage 5's own output handed back to it.
def slug_of(name):
    return "_".join("".join(ch if ch.isalnum() else " " for ch in name.lower()).split())


manifest, slug_of_run = {}, {}
for kind, group in (("ours", {LADDER[0]["name"]: ours}),
                    ("room_finder", NAME_TO_RUN),
                    ("combined", COMBINED)):
    for name, reading in group.items():
        if not reading:
            continue
        slug = slug_of(name)
        (OUT / f"{slug}.json").write_text(json.dumps(reading, indent=1))
        manifest[slug] = {"kind": kind, "name": name, "score": SCORED.get(name)}
        if kind == "room_finder":
            slug_of_run[id(reading)] = slug

# Record which room-finder run each combination grew from, so the manifest carries
# the whole chain and not just the scores.
for slug, entry in manifest.items():
    if entry["kind"] == "combined":
        entry["grown_from"] = slug_of_run.get(id(SEED_SOURCE.get(entry["name"])))
(OUT / "readings.json").write_text(json.dumps(manifest, indent=1))

# The one to import is the run whose *grown* rooms won -- growing is exactly what
# the pipeline does with them, so its score after growing is the one that counts,
# not its score on its own.
grown = [(m["score"], s) for s, m in manifest.items()
         if m["kind"] == "combined" and m["score"] is not None and m.get("grown_from")]
finders = {s: m for s, m in manifest.items()
           if m["kind"] == "room_finder" and m["score"] is not None}
pick = manifest[max(grown)[1]]["grown_from"] if grown else \
    (max(finders, key=lambda s: finders[s]["score"]) if finders else None)
if pick:
    print(f"the one to import: {pick}")
    print(f"    python -m tools.import_room_predictions results.zip --reading {pick}\n")

archive = Path(shutil.make_archive(str(ROOT / "results"), "zip", OUT))
print(f"{archive}  ({archive.stat().st_size / 1e6:.1f} MB)")
if VOLUME is not None:
    try:
        Path(VOLUME).mkdir(parents=True, exist_ok=True)
        shutil.copy(archive, Path(VOLUME) / "results.zip")
        print(f"copied to {Path(VOLUME) / 'results.zip'} — survives this machine stopping")
    except Exception as exc:                                   # noqa: BLE001
        print(f"could not copy to {VOLUME}: {exc}")
else:
    print("Download it from the file browser in the sidebar, or set VOLUME at the top of")
    print("cell 1.2 to a mounted Volume and re-run this cell to keep it.")

---

## What to do with the answer

**Feed it back into the pipeline.** The winning reading's rooms are what stage 5 wants:

```bash
python -m tools.import_room_predictions results.zip --list
python -m tools.import_room_predictions results.zip --reading <name>
python -m pipeline run <ids> --from 5
python -m tools.plan_vs_shell build --out out/review
```

The save cell above prints the exact name to use. Only the **room-finder's own rooms** can be
imported — the pipeline does the combining itself, with the same code this notebook just used,
so importing a combined reading would hand our own rooms back to us as if they were a second
opinion. The zip says which is which and `--list` shows it, so there is nothing to remember.

**If the table printed two settings**, put them in `pipeline/floorplan/vectorise.py` first.

**If nothing beat where we started**, that is a real answer. It means the room-finder is not
seeing our plans well enough for its rooms to be worth growing, and the next thing to try is
teaching a model on our own plans rather than tuning what these ones already do —
`notebooks/finetune_wallnet_colab.ipynb`.